In [25]:
from langchain_community.document_loaders import PyMuPDFLoader

file_path = "/Users/kara/Desktop/KGP-Langchain/10 Projects on PDF Documents/rag-dataset/health supplements/1. dietary supplements - for whom.pdf"
loader = PyMuPDFLoader(file_path)

In [26]:
docs = loader.load()

In [27]:
len(docs)

17

In [ ]:
print(docs[0].page_content)

In [29]:
import os

pdfs = []

for root, dirs, files in os.walk("rag-dataset"):
    for file in files:
        if file.endswith(".pdf"):
            pdfs.append(os.path.join(root,file))

In [30]:
pdfs

['rag-dataset/health supplements/1. dietary supplements - for whom.pdf',
 'rag-dataset/health supplements/3.health_supplements_side_effects.pdf',
 'rag-dataset/health supplements/2. Nutraceuticals research.pdf',
 'rag-dataset/gym supplements/2. High Prevalence of Supplement Intake.pdf',
 'rag-dataset/gym supplements/1. Analysis of Actual Fitness Supplement.pdf']

In [34]:
docs = []

for pdf in pdfs:
    loader = PyMuPDFLoader(pdf)
    docs.extend(loader.load())

In [35]:
len(docs)

64

In [37]:
def format_docs(docs):
    return "\n\n".join([x.page_content for x in docs])

context = format_docs(docs)

In [ ]:
# print(context)

In [41]:
import tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

In [43]:
len(encoding.encode(context))

58182

In [45]:
## question answering using LLM

from langchain_ollama import ChatOllama
from langchain_core.prompts import (SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate)
from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434"
model = 'qwen3:8b'


llm = ChatOllama(
    base_url=base_url,
    model=model
)


In [46]:
llm

ChatOllama(output_version=None, model='qwen3:8b', base_url='http://localhost:11434')

In [ ]:
system = SystemMessagePromptTemplate.from_template("""You are helpful AI assistant who answer user question based on the provided context. 
                                                    Do not answer in more than {words} words""")

prompt = """Answer user question based on the provided context ONLY! If you do not know the answer, just say "I don't know".
            ### Context:
            {context}

            ### Question:
            {question}

            ### Answer:"""

prompt = HumanMessagePromptTemplate.from_template(prompt)

messages = [system, prompt]
template = ChatPromptTemplate(messages)

# template
# result = template.invoke({'context': context, 'question': "How to gain muscle mass?", 'words': 50})

qna_chain = template | llm | StrOutputParser()

In [53]:
response = qna_chain.invoke({'context': context, 'question': "How to gain muscle mass?", 'words': 50})


KeyboardInterrupt



In [ ]:
response

# Project 2 - PDF document Summarizer

In [ ]:
system = SystemMessagePromptTemplate.from_template("""You are helpful AI assistant who works as document summarizer. 
                                                   You must not hallucinate or provide any false information.""")

prompt = """Summarize the given context in {words}.
            ### Context:
            {context}

            ### Summary:"""

prompt = HumanMessagePromptTemplate.from_template(prompt)

messages = [system, prompt]
template = ChatPromptTemplate(messages)

summary_chain = template | llm | StrOutputParser()

In [ ]:
summary_chain

In [ ]:
response = summary_chain.invoke({'context': context, 'words': 500})
print(response)


# Project 3 : Report Generation from PDF Document

response = qna_chain.invoke({'context': context, 
                             'question': "Provide a detailed report from the provided context. Write answer in Markdown.", 
                             'words': 2000})
print(response)